# 🔤 Byte Pair Encoding (BPE) — From Zero to Implementation

> BPE is the tokenisation algorithm behind GPT, LLaMA, Mistral, and most modern LLMs. This notebook walks you through every concept, with live animations and a full from-scratch implementation.

---

## 1. Why Do We Even Need Tokenisation?

Language models don't read words — they read **numbers**. So first, you need to turn text into a sequence of integers. The question is: *what are the chunks?*

| Approach | Example for `"running"` | Problem |
|---|---|---|
| **Word-level** | `[running]` | `run`, `running`, `runner` are treated as completely unrelated |
| **Character-level** | `[r, u, n, n, i, n, g]` | Sequences get very long, model has to learn everything from scratch |
| **Subword (BPE)** ✅ | `[run, ##ning]` | Sweet spot — shares `run` across words, still handles rare words |

BPE finds the sweet spot by learning **subword units** from data.

## 2. The Core Idea in One Sentence

> **Start with characters. Repeatedly merge the most frequent adjacent pair of tokens. Stop when you hit your target vocabulary size.**

That's it. Let's make this concrete.

## 3. What Is a Corpus?

A **corpus** is just the text you train on. A string. Nothing fancy.

```python
corpus = "low lower lowest"
```

We'll use a slightly richer one:

In [2]:
corpus = "low low low low low lower lower lowest"

## 4. Step-by-Step: How BPE Works

### Step 1 — Split corpus into words

Just split on spaces:

In [3]:
words = corpus.strip().split()
print(words)

['low', 'low', 'low', 'low', 'low', 'lower', 'lower', 'lowest']


### Step 2 — Explode each word into characters + add `</w>` marker

Every word becomes a **tuple of individual characters**.
We add `</w>` at the end to mark word boundaries — so the model knows `low` at the end of a word is different from `low` in the middle.

```
"low"    →  ('l', 'o', 'w', '</w>')
"lower"  →  ('l', 'o', 'w', 'e', 'r', '</w>')
"lowest" →  ('l', 'o', 'w', 'e', 's', 't', '</w>')
```

In [4]:
from collections import defaultdict

def get_word_freqs(corpus: str) -> dict:
    """Build word frequency table — maps char-tuple to count."""
    word_freqs = defaultdict(int)
    for word in corpus.strip().split():
        chars = tuple(list(word) + ["</w>"])
        word_freqs[chars] += 1
    return dict(word_freqs)

word_freqs = get_word_freqs(corpus)
print("Dict", word_freqs)

print("Word frequency table:")
for tokens, freq in word_freqs.items():
    print(f"  {tokens}  →  freq={freq}")

Dict {('l', 'o', 'w', '</w>'): 5, ('l', 'o', 'w', 'e', 'r', '</w>'): 2, ('l', 'o', 'w', 'e', 's', 't', '</w>'): 1}
Word frequency table:
  ('l', 'o', 'w', '</w>')  →  freq=5
  ('l', 'o', 'w', 'e', 'r', '</w>')  →  freq=2
  ('l', 'o', 'w', 'e', 's', 't', '</w>')  →  freq=1


### Step 3 — Count all adjacent pairs

For every word, look at every **neighboring pair** of tokens and tally their counts.

**Example** — `('l','o','w','</w>')` with freq=5 contributes:
```
  (l, o)    += 5
  (o, w)    += 5
  (w, </w>) += 5
```

This is the counting step. Let's implement and visualise it:

In [5]:
def get_pair_freqs(word_freqs: dict) -> dict:
    """Count how often each adjacent token pair appears across all words."""
    pair_freqs = defaultdict(int)
    for tokens, freq in word_freqs.items():
        for a, b in zip(tokens, tokens[1:]):
            pair_freqs[(a, b)] += freq
    return dict(pair_freqs)

pair_freqs = get_pair_freqs(word_freqs)

# Sort by frequency descending
sorted_pairs = sorted(pair_freqs.items(), key=lambda x: x[1], reverse=True)

print("Pair frequencies (top 10):")
for pair, freq in sorted_pairs[:10]:
    print(f"  {str(pair):25}  count={freq}")

Pair frequencies (top 10):
  ('l', 'o')                 count=8
  ('o', 'w')                 count=8
  ('w', '</w>')              count=5
  ('w', 'e')                 count=3
  ('e', 'r')                 count=2
  ('r', '</w>')              count=2
  ('e', 's')                 count=1
  ('s', 't')                 count=1
  ('t', '</w>')              count=1


### Step 4 — Merge the best pair

Take the most frequent pair and **replace every occurrence** of it across all words with the merged token.

In [ ]:
def apply_merge(word_freqs: dict, best_pair: tuple) -> dict:
    """Replace (a, b) with 'ab' everywhere in the word frequency table."""
    a, b = best_pair
    merged = a + b
    new_word_freqs = {}

    for tokens, freq in word_freqs.items():
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                new_tokens.append(merged)   # merge!
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        new_word_freqs[tuple(new_tokens)] = freq
    print(new_word_freqs)
    return new_word_freqs
    

# Apply first merge: (l, o) → lo
best_pair = sorted_pairs[0][0]
print(f"Best pair: {best_pair}  →  merging into: '{best_pair[0]+best_pair[1]}'")
print()

print("Before merge:")
for t, f in word_freqs.items(): print(f"  {t}  freq={f}")

word_freqs_after = apply_merge(word_freqs, best_pair)
print("\nAfter merge:")
for t, f in word_freqs_after.items(): print(f"  {t}  freq={f}")


Best pair: ('l', 'o')  →  merging into: 'lo'

Before merge:
  ('l', 'o', 'w', '</w>')  freq=5
  ('l', 'o', 'w', 'e', 'r', '</w>')  freq=2
  ('l', 'o', 'w', 'e', 's', 't', '</w>')  freq=1
Merged pair: {('lo', 'w', '</w>'): 5, ('lo', 'w', 'e', 'r', '</w>'): 2, ('lo', 'w', 'e', 's', 't', '</w>'): 1}

After merge:
  ('lo', 'w', '</w>')  freq=5
  ('lo', 'w', 'e', 'r', '</w>')  freq=2
  ('lo', 'w', 'e', 's', 't', '</w>')  freq=1


## 5. 🎬 Animated Step-by-Step Trace

Let's watch 10 merge iterations happen live, printing the state after each merge:

In [ ]:
import time
from IPython.display import clear_output, display

def animate_bpe(corpus: str, n_merges: int = 10, delay: float = 1.0):
    """Animate BPE merges step by step with a delay between each."""
    
    word_freqs = get_word_freqs(corpus)
    merges = []
    
    print("=" * 60)
    print("BPE ANIMATION — watch tokens collapse over time")
    print("=" * 60)
    print("\nINITIAL STATE (character-level):")
    for t, f in word_freqs.items():
        print(f"  {' '.join(t):<40}  ×{f}")
    
    time.sleep(delay)
    
    for step in range(1, n_merges + 1):
        pair_freqs = get_pair_freqs(word_freqs)
        if not pair_freqs:
            break
        
        best_pair = max(pair_freqs, key=lambda p: (pair_freqs[p], p))
        best_freq = pair_freqs[best_pair]
        if best_freq < 2:
            break
        
        merged = best_pair[0] + best_pair[1]
        merges.append(best_pair)
        word_freqs = apply_merge(word_freqs, best_pair)
        
        clear_output(wait=True)
        print("=" * 60)
        print(f"STEP {step}: Merge  '{best_pair[0]}'  +  '{best_pair[1]}'  →  '{merged}'  (freq={best_freq})")
        print("=" * 60)
        print()
        
        print("Current token state:")
        for t, f in word_freqs.items():
            # Highlight the newly merged token
            displayed = []
            for tok in t:
                if tok == merged:
                    displayed.append(f"[{tok}]")   # bracket = newly merged
                else:
                    displayed.append(tok)
            print(f"  {' '.join(displayed):<45}  ×{f}")
        
        print()
        print("All merges so far:", " → ".join(["".join(m) for m in merges]))
        
        time.sleep(delay)
    
    print()
    print("✅ Done! Final merge rules:")
    for i, (a, b) in enumerate(merges, 1):
        print(f"  Rule {i}: '{a}' + '{b}' → '{a+b}'")

# Run it! Adjust delay to slow down / speed up
animate_bpe(corpus, n_merges=10, delay=1.5)

STEP 6: Merge  'lowe'  +  'r</w>'  →  'lower</w>'  (freq=2)

Current token state:
  low</w>                                        ×5
  [lower</w>]                                    ×2
  lowe s t </w>                                  ×1

All merges so far: ow → low → low</w> → lowe → r</w> → lower</w>

✅ Done! Final merge rules:
  Rule 1: 'o' + 'w' → 'ow'
  Rule 2: 'l' + 'ow' → 'low'
  Rule 3: 'low' + '</w>' → 'low</w>'
  Rule 4: 'low' + 'e' → 'lowe'
  Rule 5: 'r' + '</w>' → 'r</w>'
  Rule 6: 'lowe' + 'r</w>' → 'lower</w>'


---

## 6. The Full BPETokenizer Class

Now let's put everything together into a clean, reusable class with **training** and **encoding**:

In [ ]:
import re
from collections import defaultdict


class BPETokenizer:
    """
    Byte Pair Encoding tokenizer — trained from scratch.

    Based on Sennrich et al. (2016). Uses end-of-word markers
    so tokens know whether they appear at word boundaries.

    Usage:
        tok = BPETokenizer(vocab_size=100)
        tok.train("your corpus here")
        tok.encode("lower")       # → ['lower</w>']
        tok.decode([...])         # → 'lower'
    """

    def __init__(self, vocab_size: int = 300):
        self.vocab_size = vocab_size
        self.merges: list[tuple[str, str]] = []   # ordered merge rules
        self.vocab: set[str] = set()

    # ── Step 1: Build word frequency table ─────────────────────────────

    def _get_word_freqs(self, corpus: str) -> dict:
        """Split corpus into words → character tuples with </w> marker."""
        word_freqs = defaultdict(int)
        for word in corpus.strip().split():
            chars = tuple(list(word) + ["</w>"])
            word_freqs[chars] += 1
        return dict(word_freqs)

    # ── Step 2: Count adjacent pairs ───────────────────────────────────

    def _get_pair_freqs(self, word_freqs: dict) -> dict:
        """Count how often each adjacent pair appears, weighted by word freq."""
        pair_freqs = defaultdict(int)
        for tokens, freq in word_freqs.items():
            for a, b in zip(tokens, tokens[1:]):
                pair_freqs[(a, b)] += freq
        return dict(pair_freqs)

    # ── Step 3: Apply a merge ──────────────────────────────────────────

    def _apply_merge(self, word_freqs: dict, best_pair: tuple) -> dict:
        """Replace every (a, b) adjacent in any word with 'ab'."""
        a, b = best_pair
        merged = a + b
        new_word_freqs = {}
        for tokens, freq in word_freqs.items():
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            new_word_freqs[tuple(new_tokens)] = freq
        return new_word_freqs

    # ── Step 4: Training loop ──────────────────────────────────────────

    def train(self, corpus: str, verbose: bool = False) -> None:
        """
        Train BPE on a corpus. Runs until vocab_size is reached
        or no more frequent pairs exist.
        """
        word_freqs = self._get_word_freqs(corpus)
        self.vocab = {char for tokens in word_freqs for char in tokens}

        if verbose:
            print(f"Initial vocab ({len(self.vocab)} tokens): {sorted(self.vocab)}")
            print(f"Target vocab size: {self.vocab_size}")
            print("-" * 50)

        num_merges = self.vocab_size - len(self.vocab)

        for i in range(num_merges):
            pair_freqs = self._get_pair_freqs(word_freqs)
            if not pair_freqs:
                break

            # Pick best pair (ties broken alphabetically for reproducibility)
            best_pair = max(pair_freqs, key=lambda p: (pair_freqs[p], p))
            best_freq = pair_freqs[best_pair]

            if best_freq < 2:
                break  # no point merging singletons

            self.merges.append(best_pair)
            new_token = best_pair[0] + best_pair[1]
            self.vocab.add(new_token)
            word_freqs = self._apply_merge(word_freqs, best_pair)

            if verbose:
                print(f"  merge {i+1:3d}: {best_pair[0]!r:10} + {best_pair[1]!r:10}"
                      f" → {new_token!r:15}  (freq={best_freq})")

        if verbose:
            print("-" * 50)
            print(f"Final vocab size: {len(self.vocab)}")
            print(f"Learned {len(self.merges)} merge rules")

    # ── Step 5: Encoding new text ──────────────────────────────────────

    def _tokenize_word(self, word: str) -> list[str]:
        """
        Tokenize a single word:
        1. Split into chars + </w>
        2. Apply merge rules IN ORDER (order matters!)
        """
        tokens = list(word) + ["</w>"]

        for a, b in self.merges:
            merged = a + b
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        return tokens

    def encode(self, text: str) -> list[str]:
        """Tokenize a full text string → list of subword tokens."""
        tokens = []
        for word in text.strip().split():
            tokens.extend(self._tokenize_word(word))
        return tokens

    def encode_ids(self, text: str) -> list[int]:
        """Like encode() but returns integer IDs."""
        vocab_list = sorted(self.vocab)
        token_to_id = {t: i for i, t in enumerate(vocab_list)}
        return [token_to_id[t] for t in self.encode(text) if t in token_to_id]

    def decode(self, tokens: list[str]) -> str:
        """Convert tokens back to text (removes </w> markers)."""
        return " ".join("".join(tokens).replace("</w>", " ").split())

print("✅ BPETokenizer class defined!")

---

## 7. Training & Testing

In [ ]:
# ── Training ────────────────────────────────────────────────────────────

corpus = """
low low low low low
lower lower lower
lowest lowest
newer new new new
"""

tokenizer = BPETokenizer(vocab_size=50)
tokenizer.train(corpus, verbose=True)

In [ ]:
# ── Encoding test words ─────────────────────────────────────────────────

test_words = ["low", "lower", "lowest", "newer", "new", "low lower"]

print("Encoding results:")
print("-" * 45)
for text in test_words:
    try:
        tokens = tokenizer.encode(text)
        print(f"  {text:<12}  →  {tokens}")
    except Exception as e:
        print(f"  {text:<12}  →  ERROR: {e}")

### 🔍 Why did `lowest` split but `lower` didn't?

- `lower` appeared **3 times** in the corpus → frequent enough to get its own merged token
- `lowest` appeared **2 times** → rarer, so it was split into `low` + `est`

**Frequency drives granularity.** Common words get collapsed into single tokens; rare words get split into reusable pieces. This is the whole point of BPE.

---

## 8. 🎨 Visual: Interactive Merge Trace

This cell prints each BPE merge as an ASCII table — you can see exactly what's happening to your tokens at each step:

In [ ]:
def trace_merges(corpus: str, n_merges: int = 15):
    """Print a full trace of BPE merges with the token table at each step."""
    
    word_freqs = get_word_freqs(corpus)
    merges = []
    
    # ── Header ──────────────────────────────────────────────────────────
    initial_vocab = sorted({c for tokens in word_freqs for c in tokens})
    print(f"Initial vocab ({len(initial_vocab)}): {initial_vocab}")
    print()
    
    col_w = 45
    print(f"{'STEP':<6} {'MERGE':<25} {'RESULT':<20} {'FREQ'}")
    print("─" * 65)
    
    for step in range(1, n_merges + 1):
        pair_freqs = get_pair_freqs(word_freqs)
        if not pair_freqs:
            break
        
        best_pair = max(pair_freqs, key=lambda p: (pair_freqs[p], p))
        freq = pair_freqs[best_pair]
        if freq < 2:
            break
        
        merged = best_pair[0] + best_pair[1]
        merges.append(best_pair)
        word_freqs = apply_merge(word_freqs, best_pair)
        
        merge_str = f"'{best_pair[0]}' + '{best_pair[1]}'"
        print(f"{step:<6} {merge_str:<25} → '{merged}'{'':10} {freq}")
    
    print("─" * 65)
    print()
    print("Final token state:")
    for t, f in word_freqs.items():
        print(f"  {' | '.join(t):<40}  ×{f}")


trace_merges(corpus, n_merges=15)

---

## 9. Three Key Concepts to Understand Deeply

### 9.1 Why merge ORDER matters for encoding

When you encode new text, you apply merges in the **exact same order** they were learned. This matters because earlier merges can consume tokens before later merges get a chance.

**Example:**
- Merge rule #3: `low` + `er` → `lower`
- Merge rule #7: `er` + `s` → `ers`

For the word `"lowers"`: rule #3 fires first, consuming `low+er` → `lower`. Now there's no bare `er` left for rule #7. Result: `[lower, s]` not `[low, ers]`.

In [ ]:
# Demonstrate order sensitivity

def encode_with_rule_subset(word: str, merges: list, use_first_n: int) -> list:
    """Encode a word using only the first N merge rules."""
    tokens = list(word) + ["</w>"]
    for a, b in merges[:use_first_n]:
        merged = a + b
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

print("How 'lower' tokenizes with increasing numbers of merge rules:")
print("-" * 50)
for n in range(0, len(tokenizer.merges) + 1, 2):
    result = encode_with_rule_subset("lower", tokenizer.merges, n)
    print(f"  After {n:2d} rules: {result}")

### 9.2 The `</w>` marker trick

Without end-of-word markers, `low` at the **end** of a word and `low` **inside** a word would produce the same token. With `</w>`, they get different representations:

- `low</w>` — the word "low" standing alone
- `low` — the prefix "low" in a longer word like "lower"

This helps the model know where words begin and end.

### 9.3 Vocabulary size is a hyperparameter

| Model | Vocab size |
|---|---|
| GPT-2 | 50,257 |
| LLaMA 2 | 32,000 |
| Mistral | 32,000 |
| GPT-4 (est.) | ~100,000 |

**Larger vocab** → fewer tokens per sentence (faster) but more parameters in the embedding layer.

**Smaller vocab** → more tokens per sentence but better generalisation to rare/unseen words.

---

## 10. Round-trip: Encode then Decode

In [ ]:
test_sentence = "low lower lowest"

tokens = tokenizer.encode(test_sentence)
ids    = tokenizer.encode_ids(test_sentence)
decoded = tokenizer.decode(tokens)

print(f"Original : {test_sentence!r}")
print(f"Tokens   : {tokens}")
print(f"Token IDs: {ids}")
print(f"Decoded  : {decoded!r}")
print()
print(f"✅ Round-trip match: {test_sentence == decoded}")

---

## 11. Save & Load the Tokenizer

In [ ]:
import json

def save_tokenizer(tokenizer: BPETokenizer, path: str):
    """Save merge rules and vocab to a JSON file."""
    data = {
        "vocab_size": tokenizer.vocab_size,
        "merges": [[a, b] for a, b in tokenizer.merges],
        "vocab": list(tokenizer.vocab)
    }
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"✅ Saved to {path}")

def load_tokenizer(path: str) -> BPETokenizer:
    """Load a tokenizer from a JSON file."""
    with open(path) as f:
        data = json.load(f)
    tok = BPETokenizer(vocab_size=data["vocab_size"])
    tok.merges = [tuple(pair) for pair in data["merges"]]
    tok.vocab = set(data["vocab"])
    print(f"✅ Loaded from {path} ({len(tok.merges)} merge rules)")
    return tok

# Save
save_tokenizer(tokenizer, "bpe_tokenizer.json")

# Reload and verify
tok2 = load_tokenizer("bpe_tokenizer.json")
print(f"Reloaded encoding of 'lower': {tok2.encode('lower')}")

---

## 12. 🚀 What's Next?

This implementation follows the original Sennrich et al. 2016 paper. Production tokenizers extend it in a few key ways:

| Extension | What it does | Used by |
|---|---|---|
| **Byte-level BPE** | Operates on raw UTF-8 bytes → no unknown tokens ever | GPT-2, GPT-4 |
| **Regex pre-tokenisation** | Splits on spaces/punctuation before BPE so numbers don't get weird merges | GPT-2 |
| **Unigram LM** | Alternative to BPE — picks tokens probabilistically | SentencePiece (T5, ALBERT) |
| **WordPiece** | Like BPE but maximises likelihood instead of frequency | BERT |

The [`tiktoken`](https://github.com/openai/tiktoken) library (OpenAI) and [`sentencepiece`](https://github.com/google/sentencepiece) (Google) are production-grade implementations worth exploring.

---

## 13. Summary Cheatsheet

```
TRAINING
--------
1. Split corpus → words → character tuples + </w>
2. Count all adjacent token pairs (weighted by word freq)
3. Pick most frequent pair
4. Merge it everywhere, add to vocab
5. Repeat until vocab_size reached

ENCODING
--------
1. Split text → words → chars + </w>
2. Apply merge rules IN ORDER (order matters!)
3. Return final token list

KEY FACTS
---------
• </w> marker distinguishes word-final vs. word-internal tokens
• Merge order is deterministic and must be preserved
• More frequent subwords → own token; rare → split into pieces
• Vocab size is a hyperparameter (GPT-2: 50k, LLaMA: 32k)
```